In [ ]:
from scipy.io import loadmat
import numpy as np
import h5py
import matplotlib.pyplot as plt
#%matplotlib notebook
%matplotlib widget

import sys
sys.path.append("../../src/utils")
from utils import *

In [ ]:
#data = loadmat("Grid_rasBase_5_nGrains_5_nRef_4_dG_3.75e-09.mat")
#data = loadmat("Grid_rasBase_5_nGrains_5_nRef_3_dG_5e-09.mat")
#data = loadmat("Grid_rasBase_5_nGrains_25_nRef_3_dG_3.75e-09.mat")
data = loadmat("Grid_rasBase_10_nGrains_25_nRef_3_dG_3.75e-09.mat")
#data = loadmat("Grid_rasBase_5_nGrains_25_nRef_4_dG_3.75e-09.mat")

#data = loadmat("Grid_rasBase_5_nGrains_5_nRef_4_dG_5e-09.mat")
#data = loadmat("Grid_rasBase_6_nGrains_25_nRef_4_dG_3.75e-09.mat")


#data = loadmat("Grid_rasBase_10_nGrains_25_nRef_1_dG_3.75e-09.mat")
#data = loadmat("Grid_rasBase_13_nGrains_25_nRef_1_dG_3.75e-09.mat")
#data = loadmat("Grid_rasBase_16_nGrains_25_nRef_1_dG_3.75e-09.mat")
#data = loadmat("Grid_rasBase_20_nGrains_25_nRef_1_dG_3.75e-09.mat")
#data = loadmat("Grid_rasBase_25_nGrains_25_nRef_1_dG_3.75e-09.mat")
#data = loadmat("Grid_rasBase_32_nGrains_25_nRef_1_dG_3.75e-09.mat")
#data = loadmat("Grid_rasBase_40_nGrains_25_nRef_1_dG_3.75e-09.mat")

def matlab_struct_to_dict(matobj):
    out = {}
    for name in matobj.dtype.names:
        elem = matobj[name][0, 0]
        if isinstance(elem, np.ndarray) and elem.dtype.names:
            out[name] = matlab_struct_to_dict(elem)
        else:
            out[name] = elem
    return out

mesh_cart = matlab_struct_to_dict(data["mesh_cart"])
GridInfo  = matlab_struct_to_dict(data["GridInfo"])
mesh_params = matlab_struct_to_dict(data["mesh_params"])

pos  = mesh_cart["pos_out"]
dims = mesh_cart["dims_out"]

print("pos shape:", pos.shape)

# iIn: convert from MATLAB cell of (Ni,1) 1-based indices → list of 1D 0-based arrays
raw_iIn = mesh_cart["iIn"][0]   # shape (n_groups,) dtypeß=object
iIn = [np.asarray(g, dtype=int).ravel() - 1 for g in raw_iIn]

fig, ax = cartesian_unstructured_mesh_plot(pos, dims, GridInfo, iIn)
plt.show()


In [ ]:
GridInfo

In [ ]:
list(mesh_params.keys())

In [ ]:
mesh_params["resolution"]

# Explanation to diffefent attributes

Nk: number of faces \
Nn: Number of tiles \
Ng: Number of grains

### GridInfo
- fNormX, fNormY, fNormZ: Normalized face direction  ( Nk )
- AreaFaces: Area of each face ( Nk )
- Volumes: Volume of each tile ( Nn )
- TheSigns: The "sign" (+-1) of each tile and face showing if it goes into our out of the cell. Sparse matrix with shape (Nn, Nk) - but often <0.1% filled
- TheTs: ???
- TheDs: ???
- Xel, Yel, Zel: Center coordinate of tile ( Nn )
- Xf, Yf, Zf: Center coordinate of face ( Nk )
- DimsF: Dimensions of each face ( Nk, 3 )

### Mesh_cart
- pos_out: Center position of each tile ( Nn, 3 )
- dims_out: Dimension of each tile ( Nn, 3 )
- children_out: ???? ( Nn, 8 )
- type: Single array value with the type of the tiles - fx. ['cartesian']
- Xc: Center position of each tile ( Nn, 3 ) - Not sure what is the difference between this and pos_out???
- iIn: List for each grain ( Ng ) containing the tiles beloning to grain index. 
- GrainIndex: Grain index for each tile ( Nn )

### mesh_params - all single value 'arrays'
- nGrains: Number of grains - fx. [[25]]
- resBase: Base resolutin - fx. [[5]]
- NNs: Similar to resBase but for each dimension - fx. [[5,5,5]]
- NumRefinments: Number of allowed refinements - fx. [[4]]
- offSetD: Size of intergranular region - fx. [[3.75e-09]]
- thisGridL: Total grid size: - fx. [[2.4e-07, 2.4e-07, 2.4e-07]]
- resolution: Resolution of grid fx. [[5,5,5]]. If unstructured it will be [[Nn, 1, 1]]


In [ ]:
GridInfo["Zel"]

In [ ]:
A = GridInfo["TheDs"].tocoo()

np.min(A.data)